In [1]:
# If not installed already (uncomment as needed):
# !pip install torch torch-geometric pandas tqdm

import os, json, torch, pandas as pd
from pathlib import Path
import importlib

import train_loop
import data_utils
importlib.reload(data_utils)
importlib.reload(train_loop)

from data_utils import load_graphs, attach_labels_and_noisy, build_loaders
from train_loop import train, evaluate
import torch, numpy as np, random
from torch_geometric.loader import DataLoader

In [2]:
# EDIT THESE:
GRAPHS_PT = "/home/macula/SMATousi/Desktop/all_6q_new/trotter/all_6_trotter_graphs_1200.pt"      # <-- Step 1 output (a list[Data])
BATCH_SIZE  = 16
NUM_WORKERS = 0
PIN_MEMORY  = torch.cuda.is_available()

In [3]:
graphs = torch.load(GRAPHS_PT, weights_only=False)   # list[Data] or InMemoryDataset
N = len(graphs)
print(f"Loaded {N} graphs")

def _set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_loaders_random(graphs, train_ratio=0.8, seed=42, batch_size=BATCH_SIZE):
    _set_seed(seed)
    idx = np.arange(len(graphs))
    np.random.shuffle(idx)
    n_train = int(len(idx) * train_ratio)
    train_idx, val_idx = idx[:n_train], idx[n_train:]
    train_set = [graphs[i] for i in train_idx]
    val_set   = [graphs[i] for i in val_idx]
    print(f"Random split: train={len(train_set)}, val={len(val_set)}")
    return (
        DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        train_idx, val_idx
    )

def build_loaders_by_indices(graphs, train_idx, val_idx, batch_size=BATCH_SIZE):
    train_set = [graphs[i] for i in train_idx]
    val_set   = [graphs[i] for i in val_idx]
    print(f"By indices: train={len(train_set)}, val={len(val_set)}")
    return (
        DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    )

def build_loaders_stratified_by_step(graphs, train_ratio=0.8, seed=42, batch_size=BATCH_SIZE):
    """Stratify by data.step (trotter_step). If missing, falls back to random."""
    # collect indices per step
    buckets = {}
    missing = []
    for i, g in enumerate(graphs):
        if hasattr(g, "step") and g.step is not None:
            s = int(float(g.step.item())) if torch.is_tensor(g.step) else int(g.step)
            buckets.setdefault(s, []).append(i)
        else:
            missing.append(i)
    if not buckets:
        print("No `step` metadata; falling back to random.")
        return build_loaders_random(graphs, train_ratio, seed, batch_size)

    _set_seed(seed)
    train_idx, val_idx = [], []
    for s, idxs in buckets.items():
        idxs = np.array(idxs)
        np.random.shuffle(idxs)
        cut = int(len(idxs) * train_ratio)
        train_idx.extend(idxs[:cut].tolist())
        val_idx.extend(idxs[cut:].tolist())
    # distribute any missing-step items proportionally
    if missing:
        miss = np.array(missing); np.random.shuffle(miss)
        cut = int(len(miss) * train_ratio)
        train_idx += miss[:cut].tolist()
        val_idx   += miss[cut:].tolist()

    train_set = [graphs[i] for i in train_idx]
    val_set   = [graphs[i] for i in val_idx]
    print(f"Stratified by step: train={len(train_set)}, val={len(val_set)}; steps={sorted(buckets.keys())}")
    return (
        DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        np.array(train_idx), np.array(val_idx)
    )

# ---- Choose ONE of the following depending on your need ----

# A) Simple random split
train_loader, val_loader, train_idx, val_idx = build_loaders_random(graphs, train_ratio=0.625, seed=123, batch_size=BATCH_SIZE)



Loaded 1200 graphs
Random split: train=750, val=450


In [7]:
model_cfg = dict(d_model=96, layers=3, heads=4, dropout=0.01637060709208519, use_noisy=True)

net, hist = train(model_cfg, (train_loader, val_loader),
                  epochs=260, 
                  lr=0.0028069921132593072, 
                  wd=1.6477856850815681e-09, 
                  patience=93, 
                  scheduler_patience=47, 
                  scheduler_factor=0,
                  best_ckpt_path="best_qem_graph_transformer.pt")


# torch.save(net.state_dict(), MODEL_OUT)
# print("Saved:", MODEL_OUT)


Epoch 001 | train 0.0632 | val 0.0633 | MAE 0.3177 | lr 2.81e-03 | new
Epoch 002 | train 0.0463 | val 0.0305 | MAE 0.2059 | lr 2.81e-03 | new
Epoch 003 | train 0.0192 | val 0.0203 | MAE 0.1779 | lr 2.81e-03 | new
Epoch 004 | train 0.0104 | val 0.0055 | MAE 0.0787 | lr 2.81e-03 | new
Epoch 005 | train 0.0057 | val 0.0047 | MAE 0.0762 | lr 2.81e-03 | new
Epoch 006 | train 0.0049 | val 0.0053 | MAE 0.0747 | lr 2.81e-03 | new
Epoch 007 | train 0.0043 | val 0.0036 | MAE 0.0637 | lr 2.81e-03 | new
Epoch 008 | train 0.0047 | val 0.0072 | MAE 0.1033 | lr 2.81e-03 | new
Epoch 009 | train 0.0051 | val 0.0047 | MAE 0.0758 | lr 2.81e-03 | new
Epoch 010 | train 0.0045 | val 0.0058 | MAE 0.0851 | lr 2.81e-03 | new
Epoch 011 | train 0.0039 | val 0.0037 | MAE 0.0665 | lr 2.81e-03 | new
Epoch 012 | train 0.0033 | val 0.0033 | MAE 0.0575 | lr 2.81e-03 | new
Epoch 013 | train 0.0032 | val 0.0038 | MAE 0.0635 | lr 2.81e-03 | new
Epoch 014 | train 0.0033 | val 0.0034 | MAE 0.0619 | lr 2.81e-03 | new
Epoch 

In [ ]:
hist

In [21]:
import torch.nn as nn
crit = nn.SmoothL1Loss()
device = "cuda" if torch.cuda.is_available() else "cpu"
metrics = evaluate(net, val_loader, crit, device=device)
metrics


{'loss': 0.0024398461299844913, 'mae': 0.05083774156040616}

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
M = next(iter(val_loader)).lightcone_masks.shape[1]

@torch.no_grad()
def collect_pred_target_noisy(loader, model, M, device):
    model.eval()
    P, Y, N = [], [], []
    for batch in loader:
        batch = batch.to(device)
        y_hat = model(batch)            # [B*M]
        P.append(y_hat.detach().cpu())
        Y.append(batch.y.detach().cpu())
        N.append(batch.noisy_z.detach().cpu())
    P = torch.cat(P, 0).view(-1, M)     # [S, M]
    Y = torch.cat(Y, 0).view(-1, M)     # [S, M]
    N = torch.cat(N, 0).view(-1, M)     # [S, M]
    return P, Y, N

P, Y, N = collect_pred_target_noisy(val_loader, net, M, device)

# ---- Per-qubit MAE (model) ----
per_qubit_mae = (P - Y).abs().mean(dim=0).tolist()
print("Per-qubit MAE (model):", per_qubit_mae)

# ---- Distances to plot ----
dist_model = (P - Y).abs()   # what you want on the plot
dist_noisy = (N - Y).abs()   # optional baseline

def plot_distance_boxplot(dist_by_source, qubit_labels=None, title="Distance to Ideal by Qubit"):
    first = next(iter(dist_by_source.values()))
    S, M = first.shape
    if qubit_labels is None:
        qubit_labels = [f"q{i}" for i in range(M)]

    fig = plt.figure(figsize=(12, 6))
    ax = plt.gca()
    base_positions = torch.arange(M).float()
    K = len(dist_by_source)
    offsets = torch.linspace(-0.25, 0.25, steps=K) if K > 1 else torch.tensor([0.0])

    handles = []
    for j, (name, D) in enumerate(dist_by_source.items()):
        data = [D[:, q].numpy() for q in range(M)]
        positions = (base_positions + offsets[j]).tolist()
        bp = ax.boxplot(
            data,
            positions=positions,
            vert=False,
            widths=0.5,
            patch_artist=False,
            manage_ticks=False,
            showfliers=False,
            whis=1.5,
            labels=None,
        )
        handles.append(bp["medians"][0])

    ax.set_yticks(base_positions.tolist())
    ax.set_yticklabels(qubit_labels)
    ax.set_xlabel("Absolute Distance (|·|)")
    ax.set_ylabel("Qubit")
    ax.set_title(title)
    ax.legend(handles, list(dist_by_source.keys()), loc="lower right", title="source")
    fig.tight_layout()
    plt.show()

# ---- Plot (model vs optional baseline) ----
plot_distance_boxplot(
    {
        "Transformer (pred−ideal)": dist_model,
        # "Noisy (noisy−ideal)": dist_noisy,   # uncomment to compare baseline
    },
    qubit_labels=[f"q{i}" for i in range(M)],
    title="Distance to Ideal Value by Qubit (Validation)",
)


In [ ]:
import json
import numpy as np

def boxplot_stats_from_tensor(D):
    """Return per-qubit boxplot statistics from [S, M] tensor."""
    stats = []
    for q in range(D.shape[1]):
        arr = D[:, q].numpy()
        q1 = np.percentile(arr, 25)
        q3 = np.percentile(arr, 75)
        median = np.median(arr)
        whisker_low = arr[arr >= q1 - 1.5 * (q3 - q1)].min()
        whisker_high = arr[arr <= q3 + 1.5 * (q3 - q1)].max()
        stats.append({
            "qbit": int(q),
            "q1": float(q1),
            "q3": float(q3),
            "median": float(median),
            "whisker_low": float(whisker_low),
            "whisker_high": float(whisker_high),
        })
    return stats

# Example: save for model distances
stats_model = boxplot_stats_from_tensor(dist_model)
stats_noisy = boxplot_stats_from_tensor(dist_noisy)

# Save to JSON
save_path = "boxplot_stats.json"
with open(save_path, "w") as f:
    json.dump({
        "Transformer (pred−ideal)": stats_model,
        "Noisy (noisy−ideal)": stats_noisy,
    }, f, indent=2)
print(f"Saved stats to {save_path}")


# Comparison

In [ ]:
import json, os, pickle
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
import qiskit.circuit.random
import torch, random
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn

import numpy as np
import json, os, pickle
from tqdm import tqdm
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from qiskit import QuantumCircuit

import sys
sys.path.append('../../../tutorials/')
from mlp import encode_data, encode_data_v2_ecr

In [ ]:
def check_f(f, f_ext, step_indices):
    return f.endswith(f_ext) and any([f"step_%02d"%step_index in f for step_index in step_indices])

def load_circuits(data_dir, step_indices, f_ext='.pk'):
    Js = []
    circuits = []
    data_paths = []
    data_files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if check_f(f, f_ext, step_indices)])
    for data_file in tqdm(data_files, leave=True):
        for entry in pickle.load(open(data_file, 'rb')):
            Js.append(entry['J'])
            circuits.append(entry['circuit'])
        data_paths.append(data_file[3:])
    return data_paths, circuits, Js

In [ ]:
data_paths, circuits, Js = load_circuits('../../../tutorials/data/ising_zne_hardware/100q_brisbane/', list(range(1, 11)))

In [ ]:
data_paths[0]

In [ ]:
for step_index in [1]:
    with open('../../../tutorials/zne_mitigated/twirl_100q_brisbane/step%02d.json'%step_index, 'r') as file:
        loaded = json.load(file)
    noise_factor_1 = np.array(loaded['noise_factor_1'])
    noise_factor_3 = np.array(loaded['noise_factor_3'])

for step_index in tqdm([2, 3, 4, 5, 6, 7, 8, 9, 10]):
    with open('../../../tutorials/zne_mitigated/twirl_100q_brisbane/step%02d.json'%step_index, 'r') as file:
        loaded = json.load(file)
    noise_factor_1 = np.concatenate([noise_factor_1, loaded['noise_factor_1']])
    noise_factor_3 = np.concatenate([noise_factor_3, loaded['noise_factor_3']])

noise_factor_1_tw_avg = noise_factor_1.reshape(noise_factor_1.shape[0], 5, 5).mean(axis=-1)
noise_factor_3_tw_avg = noise_factor_3.reshape(noise_factor_3.shape[0], 5, 5).mean(axis=-1)

slope = (noise_factor_3_tw_avg - noise_factor_1_tw_avg) / 2
zne_mitigated_vals = (noise_factor_1_tw_avg - slope).tolist()
noisy_vals = noise_factor_1_tw_avg.tolist()
len(zne_mitigated_vals)

In [ ]:
print(len(circuits), len(zne_mitigated_vals), len(noisy_vals), len(data_paths))

In [ ]:
num_circ_per_step = 50
k = train_test_split = 10
train_circuits = []
train_zne_vals = []
train_data_paths = []
train_noisy_vals = []
test_circuits = []
test_data_paths = []
test_zne_vals = []
test_noisy_vals = []
test_Js = []
for start_each_step in list(range(len(circuits))[::num_circ_per_step]):
    train_circuits += circuits[start_each_step:start_each_step+k]
    train_data_paths += data_paths[start_each_step:start_each_step+k]
    train_zne_vals += zne_mitigated_vals[start_each_step:start_each_step+k]
    train_noisy_vals += noisy_vals[start_each_step:start_each_step+k]
    test_circuits += circuits[start_each_step+k:start_each_step+num_circ_per_step]
    test_data_paths += data_paths[start_each_step+k:start_each_step+num_circ_per_step]
    test_zne_vals += zne_mitigated_vals[start_each_step+k:start_each_step+num_circ_per_step]
    test_noisy_vals += noisy_vals[start_each_step+k:start_each_step+num_circ_per_step]
    test_Js += Js[start_each_step+k:start_each_step+num_circ_per_step]

In [ ]:
print(len(train_circuits), len(train_zne_vals), len(train_noisy_vals), len(train_data_paths))
print(len(test_circuits), len(test_zne_vals), len(test_noisy_vals), len(test_data_paths))

In [ ]:
few_normal_X_train, few_normal_y_train = encode_data_v2_ecr(train_circuits, train_zne_vals, train_noisy_vals, obs_size=5)
few_normal_X_test, few_normal_y_test = encode_data_v2_ecr(test_circuits, test_zne_vals, test_noisy_vals, obs_size=5)

In [ ]:
BATCH_SIZE = 32
few_normal_train_dataset = TensorDataset(torch.Tensor(few_normal_X_train), torch.Tensor(few_normal_y_train))
few_normal_train_loader = DataLoader(few_normal_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
few_normal_test_dataset = TensorDataset(torch.Tensor(few_normal_X_test), torch.Tensor(few_normal_y_test))
few_normal_test_loader = DataLoader(few_normal_test_dataset, batch_size=BATCH_SIZE*1000, shuffle=False)

In [ ]:
few_normal_X_train = pd.DataFrame(few_normal_X_train)
few_normal_y_train = pd.DataFrame(few_normal_y_train)
few_normal_X_test = pd.DataFrame(few_normal_X_test)
few_normal_y_test = pd.DataFrame(few_normal_y_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

few_normal_rfr_tree_list = []
for q in range(5):
    rfr = RandomForestRegressor(n_estimators=100, verbose=0, n_jobs=-1)
    rfr.fit(few_normal_X_train, few_normal_y_train.iloc[:, q])
    few_normal_rfr_tree_list.append(rfr)
    print(f"Done with the {q} model")

In [ ]:
distances = []

num_spins = 5

for batch_X, batch_y in few_normal_test_loader:
    out = []
    for q, model in enumerate(few_normal_rfr_tree_list):
        out.append(model.predict(batch_X[:, :]))
    out = np.array(out).transpose()

    for ideal, noisy, ngm_mitigated in zip(
        batch_y.tolist(),
        batch_X[:, -5:].tolist(),
        out.tolist()
    ):
        for q in range(5):
            ideal_q = ideal[q]
            noisy_q = noisy[q]
            ngm_mitigated_q = ngm_mitigated[q]
            distances.append({
                f"ideal_{q}": ideal_q,
                f"noisy_{q}": noisy_q,
                f"ngm_mitigated_{q}": ngm_mitigated_q,
                f"dist_noisy_{q}": np.abs(ideal_q - noisy_q),
                f"dist_mitigated_{q}": np.abs(ideal_q - ngm_mitigated_q),
                f"dist_sq_noisy_{q}": np.square(ideal_q - noisy_q),
                f"dist_sq_mitigated_{q}": np.square(ideal_q - ngm_mitigated_q),
            })

plt.style.use({'figure.facecolor':'white'})

df = pd.DataFrame(distances)

for q in range(5):
    print(f'RMSE_noisy_{q}:', np.sqrt(df[f"dist_sq_noisy_{q}"].mean()))
    print(f'RMSE_mitigated_{q}:', np.sqrt(df[f"dist_sq_mitigated_{q}"].mean()))

print(f'RMSE_noisy:', np.sqrt(np.mean([df[f"dist_sq_noisy_{q}"].mean() for q in range(4)])))
print(f'RMSE_mitigated:', np.sqrt(np.mean([df[f"dist_sq_mitigated_{q}"].mean() for q in range(4)])))

sns.boxplot(data=df[["dist_noisy_0", "dist_mitigated_0", "dist_noisy_1", "dist_mitigated_1", "dist_noisy_2", "dist_mitigated_2", "dist_noisy_3", "dist_mitigated_3", "dist_noisy_4", "dist_mitigated_4"]], orient="h", showfliers = False)
plt.title("Dist to ideal exp value")
plt.show()

sns.histplot([df['ideal_0'], df['noisy_0'], df["ngm_mitigated_0"]], kde=True, bins=40)
plt.title("Exp values distribution")
plt.show()

In [ ]:
def evaluate_loader(test_loader, model_list, label: str, n_qbits=5):
    results = []

    for batch_X, batch_y in test_loader:
        predictions = []
        for q, model in enumerate(model_list):
            predictions.append(model.predict(batch_X[:, :]))
        predictions = np.array(predictions).transpose()

        for ideal, noisy_or_normal, mitigated in zip(
            batch_y.tolist(),
            batch_X[:, -5:].tolist(),
            predictions.tolist()
        ):
            for q in range(n_qbits):
                ideal_q = ideal[q]
                noisy_q = noisy_or_normal[q]
                mitigated_q = mitigated[q]

                results.append({
                    "source": label,
                    f"ideal_{q}": ideal_q,
                    f"input_{q}": noisy_q,
                    f"mitigated_{q}": mitigated_q,
                    f"dist_{q}": np.abs(ideal_q - noisy_q),
                    f"dist_mitigated_{q}": np.abs(ideal_q - mitigated_q),
                    f"dist_sq_{q}": np.square(ideal_q - noisy_q),
                    f"dist_sq_mitigated_{q}": np.square(ideal_q - mitigated_q),
                })
    return results

import torch
import numpy as np

def evaluate_pyg_loader(pyg_loader, net, label: str, n_qbits=5, device=None):
    """
    Produces a list[dict] with the same keys as your RF evaluate_loader().
    - 'input' = noisy_z
    - 'mitigated' = model prediction
    - 'ideal' = y (target)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    net.eval()
    results = []
    with torch.no_grad():
        for batch in pyg_loader:
            batch = batch.to(device)
            # Shapes: y_hat, y, noisy are [B*M]; we'll reshape to [B, M]
            y_hat = net(batch).detach().cpu()
            y     = batch.y.detach().cpu()
            noisy = batch.noisy_z.detach().cpu()

            # infer M from masks
            M = batch.lightcone_masks.shape[1]
            B = y_hat.numel() // M

            y_hat = y_hat.view(B, M).numpy()
            y     = y.view(B, M).numpy()
            noisy = noisy.view(B, M).numpy()

            for ideal_row, noisy_row, mitig_row in zip(y, noisy, y_hat):
                row = {"source": label}
                for q in range(n_qbits):
                    ideal_q     = float(ideal_row[q])
                    noisy_q     = float(noisy_row[q])
                    mitigated_q = float(mitig_row[q])
                    row.update({
                        f"ideal_{q}": ideal_q,
                        f"input_{q}": noisy_q,
                        f"mitigated_{q}": mitigated_q,
                        f"dist_{q}": abs(ideal_q - noisy_q),
                        f"dist_mitigated_{q}": abs(ideal_q - mitigated_q),
                        f"dist_sq_{q}": (ideal_q - noisy_q) ** 2,
                        f"dist_sq_mitigated_{q}": (ideal_q - mitigated_q) ** 2,
                    })
                results.append(row)
    return results


In [ ]:
few_normal_results = evaluate_loader(few_normal_test_loader, few_normal_rfr_tree_list, label="RF")
gnn_val_results = evaluate_pyg_loader(val_loader, net, label="QTransMLP", n_qbits=5)

all_results = few_normal_results + gnn_val_results
df = pd.DataFrame(all_results)

In [ ]:
for q in range(5):
    for label in ["RF+CLIP+LargeData", "RF+LargeData", "RF", "RF+SimulatedAndrewData", "RF+RealAndrewData", "QTransMLP"]:
        subset = df[df["source"] == label]
        rmse = np.sqrt(subset[f"dist_sq_{q}"].mean())
        rmse_mitigated = np.sqrt(subset[f"dist_sq_mitigated_{q}"].mean())
        print(f"[{label}] RMSE_input_{q}: {rmse:.4f}, RMSE_mitigated_{q}: {rmse_mitigated:.4f}")

print("------ Overall RMSEs ------")
for label in ["RF+CLIP+LargeData", "RF+LargeData", "RF", "RF+SimulatedAndrewData", "RF+RealAndrewData", "QTransMLP"]:
    subset = df[df["source"] == label]
    rmse = np.sqrt(np.mean([subset[f"dist_sq_{q}"].mean() for q in range(5)]))
    rmse_mitigated = np.sqrt(np.mean([subset[f"dist_sq_mitigated_{q}"].mean() for q in range(5)]))
    print(f"[{label}] RMSE_input: {rmse:.4f}, RMSE_mitigated: {rmse_mitigated:.4f}")



In [ ]:
melted = pd.DataFrame()

for q in range(5):
    for metric in ["dist", "dist_mitigated"]:
        temp = df[[f"{metric}_{q}", "source"]].copy()
        temp = temp.rename(columns={f"{metric}_{q}": "value"})
        temp["qubit"] = f"q{q}"
        temp["type"] = "input" if "dist_" == metric else "mitigated"
        melted = pd.concat([melted, temp], ignore_index=True)

plt.figure(figsize=(12, 6))
sns.boxplot(data=melted, x="value", y="qubit", hue="source", palette="Set2", showfliers=False)
plt.title("Distance to Ideal Value by Qubit (Input Only)")
plt.xlabel("Absolute Distance")
plt.ylabel("Qubit")
plt.savefig("RF-vs-GTrains-best.png")
plt.show()


# Mimic Results

In [ ]:
import numpy as np
import pandas as pd
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
M = 5
NUM_STEPS = 10
n_per_step = (num_circ_per_step - train_test_split)

# ---------- Helpers ----------
def _to_str_path(p):
    # Robust path normalizer for PyG batch fields
    if isinstance(p, str):
        return p
    if isinstance(p, (bytes, bytearray)):
        return p.decode()
    if isinstance(p, (list, tuple)):
        return _to_str_path(p[0]) if len(p) else ""
    return str(p)

# ---------- 1) RF/ZNE rows (keyed by circuit_path) ----------
steps_arr = np.repeat(np.arange(1, NUM_STEPS + 1), n_per_step)

models = [(few_normal_rfr_tree_list, 'rfr_list')]  # your RF list-of-trees

rf_rows = []
for circ_trans, noisy_exp_val, zne_mitigated, step, J, circ_path in tqdm(zip(
    test_circuits, test_noisy_vals, test_zne_vals, steps_arr, test_Js, test_data_paths)   # <-- include test_paths
):
    row = {
        "circuit_path": _to_str_path(circ_path),
        "step": int(step),
        "J": float(J),
        "noisy": noisy_exp_val,          # length-M vector
        "zne_mitigated": zne_mitigated,  # length-M vector (your “ideal”)
    }

    X, _ = encode_data_v2_ecr([circ_trans], [zne_mitigated], [noisy_exp_val], obs_size=M)

    for model, name in models:
        if name == 'mlp':
            row[name] = model(X).tolist()[0]
        elif name == 'gnn':
            # keep if you actually use this branch
            row[name] = model(entry.noisy_0, entry.observable, entry.circuit_depth,
                              entry.x, entry.edge_index, entry.batch).tolist()[0]
        elif name in ['ols_full', 'rfr_full']:
            row[name] = model.predict(X).tolist()[0]
        elif name in ['ols', 'rfr']:
            row[name] = model.predict(X[:, -M:]).tolist()[0]
        elif name == 'rfr_list':
            preds = [m.predict(X) for m in model]             # list of [1, M]
            row[name] = np.array(preds).transpose()[0]        # -> [M]
        elif name == 'zne':
            row[name] = zne_mitigated
        else:
            raise NotImplementedError

    rf_rows.append(row)
    # break

df_rf = pd.DataFrame(rf_rows)

# ---------- 2) Transformer rows (same split/loader), keyed by circuit_path ----------
tx_rows = []
net.eval()
with torch.no_grad():
    for batch in val_loader:  # test and val are the same set in your note
        batch = batch.to(device)
        y_hat = net(batch).detach().cpu().view(-1, M).tolist()

        # Extract per-graph circuit paths from the batch
        paths = [ _to_str_path(p)[:-4] for p in batch.circuit_path ]  # Data.circuit_path saved in conversion

        assert len(paths) == len(y_hat), "batch paths and preds length mismatch"
        for pth, pred in zip(paths, y_hat):
            tx_rows.append({
                "circuit_path": pth,
                "Transformer": pred,   # [M]
            })

df_tx = pd.DataFrame(tx_rows)

# ---------- 3) Merge by circuit_path ----------
df = pd.merge(df_rf, df_tx, on="circuit_path", how="inner")

# (Optional) quick sanity: per-step counts match after merge
counts_rf = df_rf.groupby("step").size()
counts_tx = df.groupby("step").size()
for s in sorted(counts_rf.index):
    if s in counts_tx and counts_rf[s] != counts_tx[s]:
        print(f"Warning: step {s} counts differ after merge: RF={counts_rf[s]} vs merged={counts_tx[s]}")

print("Merged rows:", len(df), "| RF rows:", len(df_rf), "| TX rows:", len(df_tx))

# ---------- 4) Plot summary (same as before; now perfectly aligned) ----------
import matplotlib.pyplot as plt

VECTOR_COLS = ["noisy", "zne_mitigated", "rfr_list", "Transformer"]

def _mean_vec(v): return float(np.mean(np.asarray(v, dtype=float)))
def _ste_series(s):
    means = s.apply(_mean_vec).values
    return float(np.std(means, ddof=0) / max(1, np.sqrt(len(means))))

def summarize_by_step(df_slice):
    gb = df_slice.groupby("step", sort=True)
    out = pd.DataFrame(index=gb.size().index)
    for col in VECTOR_COLS:
        if col in df_slice.columns:
            out[f"mean_{col}"] = gb[col].apply(lambda s: float(np.mean([_mean_vec(v) for v in s])))
            out[f"ste_{col}"]  = gb[col].apply(_ste_series)
    return out

n_samples = min(40, len(df))
for k in range(n_samples):
    # kth sample across steps -> pick rows where this is the kth element within each step block
    # since we merged by path, a stable alternative is to use sample index per step:
    df_k = df.groupby("step", sort=True).nth(k).dropna(subset=["noisy"])  # one row per step if exists
    if df_k.empty:
        continue
    summ = summarize_by_step(df_k.reset_index())
    steps_sorted = summ.index.to_list()

    plt.style.use({'figure.facecolor':'white'})
    plt.figure(figsize=(7,5))
    series_specs = []
    if "mean_noisy" in summ:          series_specs.append(("Unmitigated",          "mean_noisy",          "ste_noisy"))
    if "mean_zne_mitigated" in summ:  series_specs.append(("ZNE",                  "mean_zne_mitigated",  "ste_zne_mitigated"))
    if "mean_rfr_list" in summ:       series_specs.append(("RF Mimicking ZNE",     "mean_rfr_list",       "ste_rfr_list"))
    if "mean_Transformer" in summ:    series_specs.append(("Transformer",          "mean_Transformer",    "ste_Transformer"))

    for i, (label, mean_col, ste_col) in enumerate(series_specs):
        y = summ[mean_col].values
        se = summ[ste_col].values
        plt.plot(steps_sorted, y, label=label, marker='o', color=f"C{i+1}")
        plt.fill_between(steps_sorted, y - se, y + se, alpha=0.2, color=f"C{i+1}")

    plt.title(f'{k}-th sample across steps')
    plt.xlabel('Trotter Step'); plt.ylabel('Mean')
    xmin, xmax = plt.gca().get_xlim()
    plt.xlim([xmin, xmax]); plt.axhline(0, color='gray', linestyle='dashed')
    plt.legend(); plt.grid(False); plt.show()


In [ ]:
def summarize_by_step(df_slice: pd.DataFrame, n_per_step: int):
    """
    Group df_slice by step and compute mean and standard error
    for all vector-valued columns.
    """
    gb = df_slice.groupby("step")
    out = gb[["step"]].first().copy()

    # Loop over all columns in this slice (besides 'step')
    for col in df_slice.columns:
        if col == "step":
            continue
        # Some cols are arrays (like rfr_list, transformer), so wrap with np.mean
        out[f"mean_{col}"] = gb[col].apply(lambda x: np.mean(np.vstack(x), axis=0) if isinstance(x.iloc[0], (list, np.ndarray)) else x.mean())
        out[f"ste_{col}"] = gb[col].apply(lambda x: np.std(np.vstack(x), axis=0)/np.sqrt(n_per_step) if isinstance(x.iloc[0], (list, np.ndarray)) else x.std()/np.sqrt(n_per_step))

    return out


In [ ]:
# --- Average curves across ALL samples (scalar per step) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# methods you may have; present ones will be plotted
CANDIDATES = ["noisy", "zne_mitigated", "rfr_list", "Transformer", "transformer"]

def _row_scalar(v):
    """
    v can be a list/np array of length M (per-qubit values) or a scalar.
    Return a scalar: mean over qubits for vectors; float for scalars.
    """
    if isinstance(v, (list, tuple, np.ndarray)):
        arr = np.asarray(v, dtype=float)
        return float(arr.mean())
    # already scalar
    return float(v)

def summarize_global_scalar(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each step and each available method:
      1) Convert each row's vector to a scalar via mean over qubits.
      2) Compute mean and standard error across rows for that step.
    Returns wide dataframe indexed by step with mean_*/ste_* columns (scalars).
    """
    if "step" not in df.columns:
        raise KeyError("df must contain a 'step' column.")
    gb = df.groupby("step", sort=True)
    out = pd.DataFrame(index=sorted(df["step"].unique()))
    for col in CANDIDATES:
        if col not in df.columns: 
            continue
        # series of per-row scalars for each group
        per_step_scalars = gb[col].apply(lambda s: np.array([_row_scalar(x) for x in s]))
        # mean/ste over rows for each step
        out[f"mean_{col}"] = per_step_scalars.apply(lambda a: float(a.mean()) if a.size else np.nan)
        out[f"ste_{col}"]  = per_step_scalars.apply(lambda a: float(a.std(ddof=0)/np.sqrt(max(1,len(a)))) if a.size else np.nan)
    return out

# Build the summary on the SAME df_k you’re using
# (df_k already has 'step' and columns like 'noisy', 'rfr_list', 'Transformer')
summ = summarize_global_scalar(df)

# ---- Plot: one figure, one line per available method ----
plt.style.use({'figure.facecolor':'white'})
plt.figure(figsize=(7,5))

legend_map = {
    "mean_noisy": "Unmitigated",
    "mean_zne_mitigated": "ZNE",
    "mean_rfr_list": "RF Mimicking ZNE",
    "mean_Transformer": "Transformer",
    "mean_transformer": "Transformer",  # handle lowercase key
}

x = summ.index.values
i = 0
y_zne   = summ["mean_zne_mitigated"].values.astype(float)

for mean_col in [c for c in summ.columns if c.startswith("mean_")]:
    ste_col = mean_col.replace("mean_", "ste_")
    if ste_col not in summ.columns:
        continue
    y   = summ[mean_col].values.astype(float)
    ste = summ[ste_col].values.astype(float)
    label = legend_map.get(mean_col, mean_col.replace("mean_", ""))
    if label == "Transformer":
        label = "QTransMLP"

    # skip entirely-NaN series
    if np.isnan(y).all():
        continue

    plt.plot(x, y, marker='o', label=label, color=f"C{i}")
    plt.fill_between(x, y - ste, y + ste, alpha=0.2, color=f"C{i}")
    i += 1

plt.title("Average Across All Samples (per step)")
plt.xlabel("Trotter Step")
plt.ylabel("Mean Expected Values (avg over qubits)")
plt.axhline(0, color="gray", linestyle="dashed")
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.savefig("trotter-steps-comparison.png")
plt.show()
